# Deep Learning

### *Nina Haide*

## Introduction

This notebook represents the deep learning component of the portfolio, building directly on the previous clustering project for DACH AI job postings by keyword-extracted skill profiles. It asks the same question at a deeper level of representation: Can we discover meaningful AI job market personas from the semantic content of job descriptions using a Transformer-based model?

Rather than matching predefined keywords, this project encodes full job description text using DistilBERT, a pre-trained Transformer model and applies clustering to the resulting dense semantic vectors. The experimental comparison tests whether Transformer embeddings produce more coherent and meaningful role archetypes than a classical TF-IDF baseline - directly demonstrating the value of deep learning representations over traditional text vectorisation. The geographic scope is expanded beyond DACH to include the United Kingdom, France and the Netherlands,providing a broader European perspective on AI role archetypes.

### Imports

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

In [2]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 3.7 MB/s  0:00:21 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 3.9 MB/s  0:00:00m 3.9 MB/s eta 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.0
    Uninstalling setuptools-82.0.0:
      Successfully uninstalled setuptools-82.0.0━━━━━━ 0/2 [setuptools]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torch]━━━━━ 1/2 [torch]


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

## Data Set

The dataset was created using a custom collection pipeline built on the Adzuna
Jobs API, extended from the DACH-focused scripts developed previous projects.
The collection script `fetch_ai_jobs_personas.py` queries AI job postings
across six countries - Germany, United Kingdom, France, Netherlands, Austria and
Switzerland - using 30 targeted search queries spanning five role categories:
AI Engineering, Data Analytics, AI Product and Project Management, AI Integration
and AI Consulting. English-language queries run across all countries; German-language
queries are restricted to DACH.

Because the Adzuna API truncates descriptions at approximately 500 characters,
a secondary enrichment step (`enrich_descriptions.py`) follows each posting's
redirect URL, fetches the full job page from the source website and replaces the
truncated text with the complete description. The final dataset is stored in
`adzuna_ai_jobs_europe_enriched.csv`. After filtering out postings where enrichment
failed, the modeling dataset contains job postings with full descriptions used as
input to the DistilBERT encoder. The United Kingdom and Germany account for the
majority of entries (675 and 409 respectively), with France, Netherlands, Austria
and Switzerland contributing the remainder.

In [7]:
df = pd.read_csv("adzuna_ai_jobs_europe.csv")
df.head()

,id,title,company,location,country,country_code,category,contract_type,contract_time,salary_min,salary_max,date_posted,description,role_category,search_query,url,desc_length
0,de_5708353171,Senior Engineer Machine Learning / AI (w/m/d),HUK-COBURG Versicherungsgruppe,"Coburg, Coburg (Kreis)",Germany,de,Technikerstellen,NaN,NaN,NaN,NaN,2026-04-23,"Hallo, lass uns WIR sagen! Wir - die HUK-COBUR...",AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/land/ad/5708353171?se=Io...,500
1,de_5722278756,AI & Machine Learning Engineer (all genders),SThree Germany,"Hausen, Frankfurt am Main",Germany,de,IT-Stellen,permanent,full_time,80000.0,95000.0,2026-05-08,AI & Machine Learning Engineer (all genders) U...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5722278756?utm_m...,500
2,de_5680334179,Digital Process Engineer – AI & Automation (KI...,GPE MedTech Systeme GmbH,"Dassow, Nordwestmecklenburg (Kreis)",Germany,de,Beraterstellen,NaN,NaN,NaN,NaN,2026-03-27,Digital Process Engineer – AI & Automation (KI...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5680334179?utm_m...,500
3,de_5418313407,Senior AI / Machine Learning Engineer – Enterp...,Nortal,Deutschland,Germany,de,IT-Stellen,permanent,full_time,NaN,NaN,2025-09-27,Wir bei Nortal unterstützen den öffentlichen S...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5418313407?utm_m...,500
4,de_5698582069,AI & Machine Learning Engineer – Sensor Fusion...,Collective.work,"Altstadt-Lehel, München",Germany,de,Sonstige/Allgemeine Stellen,permanent,full_time,65000.0,105000.0,2026-04-14,Job description Salary: 65-105k€ (Annuel) AI &...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5698582069?utm_m...,500


In [ ]:
pip freeze > requirements.txt